# Spotify Streaming Audit

## Problema de negocio

Una plataforma de streaming musical cuenta con un catálogo de **1.994 canciones** sin criterios claros para decidir cuáles deberían formar parte de sus playlists premium.

El análisis preliminar muestra que una parte importante del catálogo tiene bajo rendimiento en términos de `popularity`, lo que puede afectar:

- la experiencia del usuario,
- la percepción de calidad del servicio premium,
- la retención,
- y el engagement.

---

## Objetivo del proyecto

El objetivo es identificar qué canciones, géneros y segmentos musicales deberían priorizarse para playlists premium usando análisis de datos y machine learning.

---

## Pregunta de negocio

**¿Qué canciones del catálogo deberían formar parte de playlists premium y bajo qué criterios debería tomarse esta decisión de forma sostenible?**

---

## Dataset utilizado

Se utilizará el dataset original disponible en el repositorio:

`data/raw/Spotify-2000.csv`

El notebook descarga el archivo directamente desde GitHub usando su enlace `raw`, por lo que puede ejecutarse en Jupyter Notebook, VS Code o cualquier entorno con conexión a internet, sin depender de Google Colab ni de rutas locales del computador.

# 1. Importación de librerías

Se importan las librerías necesarias para:

- limpieza de datos,
- análisis exploratorio,
- visualización,
- SQL,
- machine learning,
- clustering,
- y exportación de resultados.

In [ ]:
# ============================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================

import pandas as pd
import numpy as np
import re
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

# Carpetas de trabajo
DATA_DIR = Path("data")
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
VISUALIZATIONS_DIR = Path("visualizations")

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATIONS_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías cargadas correctamente.")
print("Carpeta raw:", RAW_DATA_DIR)
print("Carpeta processed:", PROCESSED_DATA_DIR)

# 2. Carga del dataset original de Spotify

El dataset original se encuentra en el repositorio dentro de:

`data/raw/Spotify-2000.csv`

Para que el notebook pueda ejecutarse en Jupyter Notebook, VS Code o cualquier entorno con conexión a internet, el archivo se carga directamente desde GitHub usando su enlace `raw`.



In [ ]:
# ============================================
# RUTA RAW DEL DATASET ORIGINAL EN GITHUB
# ============================================

RAW_SPOTIFY_URL = "https://raw.githubusercontent.com/isaacnhf-oss/Spotify-Streaming-Audit-/main/data/raw/Spotify-2000.csv"

print("URL del dataset original:")
print(RAW_SPOTIFY_URL)

### Resultado esperado

Pandas debe cargar correctamente el archivo `Spotify-2000.csv` desde GitHub y mostrar sus dimensiones iniciales.

Esta carga funciona en Jupyter Notebook y VS Code sin necesidad de subir archivos manualmente.

In [ ]:
# ============================================
# CARGA DEL DATASET ORIGINAL
# ============================================

df_raw = pd.read_csv(RAW_SPOTIFY_URL)

print("Dataset cargado correctamente.")
print("Dimensiones del dataset original:")
print(df_raw.shape)

display(df_raw.head())

# 3. Limpieza y estandarización del dataset

Para que el notebook sea reproducible y profesional, se realizará una limpieza inicial:

- nombres de columnas en formato `snake_case`,
- textos en minúsculas,
- eliminación de espacios innecesarios,
- conversión de duración a formato numérico,
- revisión de nulos,
- revisión de duplicados,
- exportación del dataset limpio a `data/processed/spotify_2000_clean.csv`.

El archivo limpio generado será utilizado posteriormente por el notebook principal del proyecto.

In [ ]:
def to_snake_case(column_name):
    column_name = column_name.strip().lower()
    column_name = column_name.replace("(bpm)", "bpm")
    column_name = column_name.replace("(db)", "db")
    column_name = column_name.replace("(duration)", "duration")
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name


df = df_raw.copy()
df.columns = [to_snake_case(col) for col in df.columns]

df.columns

### Resultado esperado

Los nombres de columnas quedan en formato limpio:

- `title`
- `artist`
- `top_genre`
- `year`
- `beats_per_minute_bpm`
- `energy`
- `danceability`
- `loudness_db`
- `liveness`
- `valence`
- `length_duration`
- `acousticness`
- `speechiness`
- `popularity`

In [ ]:
# ============================================
# NORMALIZACIÓN DE COLUMNAS DE TEXTO
# ============================================

text_columns = ["title", "artist", "top_genre"]

for col in text_columns:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )

display(df[[col for col in text_columns if col in df.columns]].head())

In [ ]:
# ============================================
# CONVERSIÓN DE DURACIÓN A VARIABLE NUMÉRICA
# ============================================

if "length_duration" in df.columns:
    df["length_duration"] = (
        df["length_duration"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df["length_duration"] = pd.to_numeric(
        df["length_duration"],
        errors="coerce"
    )

    display(df["length_duration"].head())
else:
    print("La columna length_duration no existe en el dataset.")

In [ ]:
# ============================================
# REVISIÓN DE NULOS Y DUPLICADOS
# ============================================

print("Valores nulos por columna:")
display(df.isnull().sum())

duplicados_antes = df.duplicated().sum()

print()
print("Duplicados antes de limpieza:")
print(duplicados_antes)

In [ ]:
# ============================================
# ELIMINACIÓN DE DUPLICADOS
# ============================================

filas_antes = df.shape[0]

df = df.drop_duplicates().reset_index(drop=True)

filas_despues = df.shape[0]
duplicados_eliminados = filas_antes - filas_despues

print("Dimensiones después de eliminar duplicados:")
print(df.shape)

print("Duplicados eliminados:")
print(duplicados_eliminados)

In [ ]:
# ============================================
# EXPORTACIÓN DEL DATASET LIMPIO
# ============================================

output_path = PROCESSED_DATA_DIR / "spotify_2000_clean.csv"

df.to_csv(
    output_path,
    index=False,
    sep=",",
    encoding="utf-8"
)

print("Dataset limpio exportado correctamente.")
print("Ruta de salida:", output_path)
print("Dimensiones exportadas:", df.shape)

### Conclusión de limpieza

El dataset de Spotify quedó estandarizado y preparado para el análisis principal:

- columnas en formato `snake_case`,
- textos normalizados,
- duración convertida a variable numérica,
- duplicados revisados y eliminados,
- dataset limpio exportado en `data/processed/spotify_2000_clean.csv`.

A partir de este punto se trabajará con `df`, que representa la versión limpia del dataset.

# 4. Exploración inicial del dataset

Se revisa la estructura general del catálogo.

In [ ]:
df.info()

In [ ]:
df.describe()

### Hallazgo inicial

El dataset contiene variables útiles para evaluar canciones:

- `popularity`: métrica principal de rendimiento.
- `top_genre`: género musical.
- `year`: año de lanzamiento.
- `energy`, `danceability`, `loudness_db`, `acousticness`, `speechiness`: atributos acústicos.

La variable objetivo para el análisis y modelado será:

## `popularity`

El archivo limpio exportado desde este notebook será utilizado por el notebook principal del proyecto para EDA, SQL, regresión, clustering e integración con Grammy Awards.